# Populate the venues table from the supplied raw data

Uses your supplied Overpass JSON, preserved byte-for-byte in `raw/official/venues/supplied_venues_20261017.json`. Run the area, postcode and business notebooks first. The table uses the same six columns and database conventions as the other notebooks: `id`, `area_id`, `business_id`, `postcode_id`, `address_line`, `size`.

IDs include the OSM object type. Area and postcode links use exact postcode matches; business links require the verified mapping CSV. Size means explicitly mapped total floor area in square metres. Unknown values stay NULL. This is a candidate inventory: buildings and businesses may represent the same physical premises. No council-boundary completeness claim is made for this attachment because it contains no boundary record or query provenance.

Raw records, tags, names and coordinates are displayed before any database write and exported as CSV. Attribution: Â© OpenStreetMap contributors; Open Database Licence (https://www.openstreetmap.org/copyright).

## 1. Settings

Paths work from the repository root or data directory. This notebook reads the supplied local snapshot and makes no network requests. Database writes occur only in section 6.

In [ ]:
import csv
import hashlib
import json
import math
import re
import sqlite3
from collections import Counter
from contextlib import closing
from datetime import datetime, timezone
from pathlib import Path

cwd = Path.cwd()
DATA_DIR = cwd / "data" if (cwd / "data" / "populate_area.ipynb").is_file() else cwd
if not (DATA_DIR / "populate_area.ipynb").is_file():
    raise RuntimeError("Run from the repository root or data directory.")
RAW_DIR = DATA_DIR / "raw" / "official" / "venues"
RAW_PATH = RAW_DIR / "supplied_venues_20261017.json"
LINKS_PATH = RAW_DIR / "venue_business_links.csv"
REPORT_DIR = DATA_DIR / "processed" / "venues_import"
DB_PATH = DATA_DIR / "processed" / "database.db"


## 2. Inspect and validate the supplied raw data

The first ten complete records appear below. All source records are exported to raw_records.csv for inspection, including tags as JSON. The original attached bytes remain available in RAW_PATH. The input hash identifies this attachment; the older downloaded snapshot and its metadata are not used.

In [ ]:
raw_bytes = RAW_PATH.read_bytes()
payload = json.loads(raw_bytes.decode("utf-8-sig"))
if payload.get("remark"):
    raise ValueError(f"Overpass reported a possible incomplete response: {payload['remark']}")
elements = payload.get("elements")
if not isinstance(elements, list) or not elements:
    raise ValueError("Expected a nonempty Overpass elements list.")
seen = set()
for el in elements:
    if el.get("type") not in {"node", "way", "relation"} or type(el.get("id")) is not int or el["id"] <= 0:
        raise ValueError("Invalid OSM identifier.")
    key = f"{el['type']}/{el['id']}"
    if key in seen:
        raise ValueError(f"Duplicate OSM identifier: {key}")
    seen.add(key)
    tags = el.get("tags")
    if not isinstance(tags, dict) or not tags or any(not isinstance(v, str) for v in tags.values()):
        raise ValueError(f"Invalid tags: {key}")
source_metadata = {
    "source": "User attachment: message (1).txt",
    "sha256": hashlib.sha256(raw_bytes).hexdigest(),
    "osm_base_timestamp": payload.get("osm3s", {}).get("timestamp_osm_base"),
    "attribution": "Â© OpenStreetMap contributors",
    "licence": "https://www.openstreetmap.org/copyright",
    "scope": "As supplied; query and council boundary not independently established",
}
print(f"Raw file: {RAW_PATH.resolve()}")
print(f"{len(elements):,} records; types: {dict(Counter(el['type'] for el in elements))}")
print(json.dumps(elements[:10], indent=2, ensure_ascii=False))
REPORT_DIR.mkdir(parents=True, exist_ok=True)
raw_csv_path = REPORT_DIR / ("raw_records_" + source_metadata["sha256"][:16] + ".csv")
with raw_csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["venue_id", "name", "lat", "lon", "tags_json", "raw_record_json"])
    for el in elements:
        point = el if el["type"] == "node" else el.get("center", {})
        writer.writerow([f"{el['type']}/{el['id']}", el["tags"].get("name", ""),
                         point.get("lat"), point.get("lon"),
                         json.dumps(el["tags"], ensure_ascii=False), json.dumps(el, ensure_ascii=False)])
print(f"All raw records exported to: {raw_csv_path.resolve()}")

## 3. Read the database and optional company mappings

A postcode match supplies both PostcodeID and AreaID; no nearest-postcode or guessed LSOA assignment is used. The postcode directory's best-fit LSOA is an approximation for the whole postcode, not an exact point-in-polygon venue assignment. Blank, invalid, multiple or unavailable postcode values leave both links NULL and appear in the review report. A matched postcode outside the loaded Southampton areas is also left unlinked.

**Business links:** edit the supplied `venue_business_links.csv` with columns `venue_id,business_id`, for example `node/123456,00123456`, only after verifying the company operating that particular venue. The file starts with a header only. Every supplied company must already exist in `business`; every venue must exist in this snapshot. Trading names, brand names, registered-office addresses and proximity do not establish legal ownership or operation, so there is no automatic name match. One venue can link to one company in this schema.


In [ ]:
if not DB_PATH.is_file():
    raise FileNotFoundError("Run the area, postcode and business notebooks first.")
with closing(sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)) as con:
    required_tables = {"area", "postcode", "business"}
    tables = {row[0] for row in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    if not required_tables <= tables:
        raise ValueError(f"Run prerequisite notebooks. Missing tables: {required_tables - tables}")
    area_ids = {row[0] for row in con.execute("SELECT id FROM area WHERE lad_code = 'E06000045'")}
    postcodes = {row[0]: (row[1], row[2]) for row in con.execute("SELECT postcode, id, area_id FROM postcode")}
    businesses = {row[0] for row in con.execute("SELECT id FROM business")}
if not area_ids or not postcodes or not businesses:
    raise ValueError("Load Southampton areas, postcodes and businesses before importing venues.")

def normalize_postcode(value):
    value = re.sub(r"\s+", "", value.upper())
    if not re.fullmatch(r"[A-Z]{1,2}\d[A-Z\d]?\d[A-Z]{2}", value):
        return None
    return value[:-3] + " " + value[-3:]

normalized_postcodes = {}
for postcode, ids in postcodes.items():
    key = normalize_postcode(postcode)
    if key is None or key in normalized_postcodes:
        raise ValueError(f"Invalid or duplicate normalized postcode in database: {postcode}")
    normalized_postcodes[key] = ids

venue_ids = {f"{el['type']}/{el['id']}" for el in elements}
business_links = {}
if LINKS_PATH.exists():
    with LINKS_PATH.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ["venue_id", "business_id"]:
            raise ValueError("Business mapping CSV needs exactly venue_id,business_id.")
        for row in reader:
            if None in row or row.get("venue_id") is None or row.get("business_id") is None:
                raise ValueError("Malformed business mapping row.")
            venue_id, business_id = row["venue_id"].strip(), row["business_id"].strip().upper()
            if venue_id not in venue_ids or business_id not in businesses:
                raise ValueError(f"Unknown venue or business in mapping: {row}")
            if venue_id in business_links:
                raise ValueError(f"Repeated venue mapping: {venue_id}")
            business_links[venue_id] = business_id
print(f"{len(business_links)} verified venue-to-business links supplied.")


## 4. Prepare addresses and floor areas

AddressLine uses `addr:full` when supplied; otherwise available unit, building name, house number/street or place, suburb and city components are joined. Missing components are not filled from nearest features.

Size uses only [`building:floor_area`](https://wiki.openstreetmap.org/wiki/Key:building:floor_area), interpreted as the mapped object's total floor area in square metres. Bare numbers, m2, mÂ² and sqm are accepted; missing, invalid or other-unit values become NULL and are reported. **A building's total area is not necessarily the floor area occupied by a particular shop or tenant.** No size is derived from polygons, number of storeys, land area or a nearby/shared building. OSM floor-area coverage may be very sparse.


In [ ]:
def address_line(tags):
    if tags.get("addr:full", "").strip():
        return " ".join(tags["addr:full"].split())
    parts = []
    if tags.get("addr:unit", "").strip():
        parts.append("Unit " + tags["addr:unit"].strip())
    if tags.get("addr:housename", "").strip():
        parts.append(tags["addr:housename"].strip())
    street = " ".join(filter(None, [
        tags.get("addr:housenumber", "").strip(),
        (tags.get("addr:street") or tags.get("addr:place") or "").strip()]))
    if street:
        parts.append(street)
    for key in ("addr:suburb", "addr:city"):
        value = tags.get(key, "").strip()
        if value and value not in parts:
            parts.append(value)
    return ", ".join(parts) or None

def floor_area(tags):
    value = tags.get("building:floor_area", "").strip()
    if not value:
        return None, "floor_area_missing"
    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*(?:m2|mÂ²|sqm)?", value, re.IGNORECASE)
    if not match:
        return None, "floor_area_invalid_or_unsupported_unit"
    number = float(match.group(1))
    if not math.isfinite(number) or number <= 0:
        return None, "floor_area_invalid_or_unsupported_unit"
    return number, None

rows = []
review = []
for el in elements:
    venue_id = f"{el['type']}/{el['id']}"
    tags = el["tags"]
    reasons = []
    area_id = postcode_id = None
    raw_postcode = (tags.get("addr:postcode") or tags.get("contact:postcode") or "").strip()
    postcode = normalize_postcode(raw_postcode) if raw_postcode else None
    if not raw_postcode:
        reasons.append("postcode_missing")
    elif postcode is None:
        reasons.append("postcode_invalid_or_multiple")
    elif postcode not in normalized_postcodes:
        reasons.append("postcode_not_in_database")
    else:
        candidate_postcode_id, candidate_area_id = normalized_postcodes[postcode]
        if candidate_area_id in area_ids:
            postcode_id, area_id = candidate_postcode_id, candidate_area_id
        else:
            reasons.append("postcode_outside_loaded_southampton")
    business_id = business_links.get(venue_id)
    if business_id is None:
        reasons.append("business_link_unverified")
    address = address_line(tags)
    if address is None:
        reasons.append("address_missing")
    size, size_reason = floor_area(tags)
    if size_reason:
        reasons.append(size_reason)
    rows.append((venue_id, area_id, business_id, postcode_id, address, size))
    if reasons:
        review.append({
            "venue_id": venue_id, "name": tags.get("name", ""),
            "osm_url": f"https://www.openstreetmap.org/{el['type']}/{el['id']}",
            "raw_postcode": raw_postcode, "area_id": area_id, "postcode_id": postcode_id,
            "business_id": business_id, "address_line": address,
            "raw_floor_area": tags.get("building:floor_area", ""),
            "size_m2": size, "reason": ";".join(reasons),
        })

print(f"{len(rows):,} venue rows prepared")
print(f"{sum(r[1] is not None for r in rows):,} linked to an area and postcode")
print(f"{sum(r[2] is not None for r in rows):,} linked to a verified business")
print(f"{sum(r[4] is not None for r in rows):,} have address information")
print(f"{sum(r[5] is not None for r in rows):,} have usable mapped floor area")
rows[:5]


## 5. Save the review report and import manifest

All venue candidates are retained, including incomplete records. The review report exposes missing attributes and links. Raw source data stays in `data/raw/official/venues/`; generated reports go in the Git-ignored `data/processed/venues_import/`.


In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
review_path = REPORT_DIR / f"venue_review_{run_id}.csv"
fields = ["venue_id", "name", "osm_url", "raw_postcode", "area_id", "postcode_id",
          "business_id", "address_line", "raw_floor_area", "size_m2", "reason"]
with review_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(review)
manifest = {
    "status": "prepared", "prepared_at_utc": datetime.now(timezone.utc).isoformat(),
    "source": source_metadata, "raw_file": str(RAW_PATH.resolve()), "row_count": len(rows),
    "business_mapping_sha256": hashlib.sha256(LINKS_PATH.read_bytes()).hexdigest() if LINKS_PATH.exists() else None,
    "size_definition": "Mapped object total floor area from building:floor_area, square metres",
    "area_policy": "Exact normalized postcode match to loaded Southampton area via postcode.area_id",
    "business_policy": "Verified mapping only; preserve existing business link if no mapping supplied",
    "review_file": str(review_path.resolve()),
}
manifest_path = REPORT_DIR / f"import_{run_id}.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(review_path)


## 6. Create and populate venues

This cell writes to the existing database. Schema creation, upserts and verification share a transaction. IDs stay stable across imports of the same OSM objects. Objects removed or retagged in OSM are not automatically deleted; review them separately. This is an accumulating inventory, not a guaranteed current operational-status register.

Source-derived address, postcode, area and size are refreshed, including NULL when missing in the snapshot. An existing BusinessID is retained when the mapping file supplies no replacement; clearing a verified business link requires an explicit update. Non-NULL foreign keys must resolve. Postcode and area consistency is checked before commit. Existing incompatible schemas fail rather than being replaced.


In [ ]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS venues (
    id           TEXT NOT NULL PRIMARY KEY,
    area_id      INTEGER REFERENCES area(id),
    business_id  TEXT REFERENCES business(id),
    postcode_id  INTEGER REFERENCES postcode(id),
    address_line TEXT,
    size         REAL CHECK (size IS NULL OR size > 0)
)
"""
UPSERT = """
INSERT INTO venues (id, area_id, business_id, postcode_id, address_line, size)
VALUES (?, ?, ?, ?, ?, ?)
ON CONFLICT(id) DO UPDATE SET
    area_id = excluded.area_id,
    business_id = COALESCE(excluded.business_id, venues.business_id),
    postcode_id = excluded.postcode_id,
    address_line = excluded.address_line,
    size = excluded.size
"""
with closing(sqlite3.connect(DB_PATH, timeout=30)) as con:
    con.execute("PRAGMA foreign_keys = ON")
    try:
        con.execute("BEGIN IMMEDIATE")
        # Check that the postcode lookup used during preparation is still current.
        if {r[0]: (r[1], r[2]) for r in con.execute("SELECT postcode, id, area_id FROM postcode")} != postcodes:
            raise RuntimeError("Postcode table changed; rerun preparation.")
        if {r[0] for r in con.execute("SELECT id FROM area WHERE lad_code='E06000045'")} != area_ids:
            raise RuntimeError("Southampton areas changed; rerun preparation.")
        con.execute(SCHEMA)
        actual_columns = [(r[1], r[2].upper(), r[5]) for r in con.execute("PRAGMA table_info(venues)")]
        expected_columns = [
            ("id", "TEXT", 1), ("area_id", "INTEGER", 0), ("business_id", "TEXT", 0),
            ("postcode_id", "INTEGER", 0), ("address_line", "TEXT", 0), ("size", "REAL", 0)]
        if actual_columns != expected_columns:
            raise ValueError(f"Existing venues schema differs: {actual_columns}")
        fk = {(r[3], r[2], r[4]) for r in con.execute("PRAGMA foreign_key_list(venues)")}
        if not {("area_id", "area", "id"), ("business_id", "business", "id"),
                ("postcode_id", "postcode", "id")} <= fk:
            raise ValueError("Existing venues table is missing expected foreign keys.")
        previous_businesses = dict(con.execute("SELECT id, business_id FROM venues"))
        for column in ("area_id", "business_id", "postcode_id"):
            con.execute(f"CREATE INDEX IF NOT EXISTS idx_venues_{column} ON venues({column})")
        con.executemany(UPSERT, rows)
        saved = {r[0]: r for r in con.execute(
            "SELECT id, area_id, business_id, postcode_id, address_line, size FROM venues")}
        for venue_id, area_id, business_id, postcode_id, address, size in rows:
            effective_business = business_id if business_id is not None else previous_businesses.get(venue_id)
            expected = (venue_id, area_id, effective_business, postcode_id, address, size)
            if saved.get(venue_id) != expected:
                raise RuntimeError(f"Stored row differs for {venue_id}")
        violations = con.execute("PRAGMA foreign_key_check(venues)").fetchall()
        if violations:
            raise RuntimeError(f"Broken venues foreign keys: {violations[:5]}")
        conflicts = con.execute("""
            SELECT v.id FROM venues v JOIN postcode p ON p.id = v.postcode_id
            WHERE v.area_id IS NOT p.area_id LIMIT 5
        """).fetchall()
        if conflicts:
            raise RuntimeError(f"Venue area disagrees with its postcode: {conflicts}")
        con.commit()
    except Exception:
        con.rollback()
        raise

manifest["status"] = "committed"
manifest["committed_at_utc"] = datetime.now(timezone.utc).isoformat()
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Loaded and verified {len(rows):,} venue candidates into {DB_PATH}")
print(f"Manifest: {manifest_path}")


## 7. Inspect the loaded inventory

Use the raw OSM snapshot or review report to see venue names and categories, which are outside the requested six-column table. Count physical premises only after reviewing duplicate OSM representations. Missing links and sizes mean unknown, not zero.


In [ ]:
with closing(sqlite3.connect(DB_PATH.resolve().as_uri() + "?mode=ro", uri=True)) as con:
    print("Total venues; linked areas; linked companies; linked postcodes; known floor areas:")
    print(con.execute("""
        SELECT COUNT(*), COUNT(area_id), COUNT(business_id), COUNT(postcode_id), COUNT(size)
        FROM venues
    """).fetchone())
    for row in con.execute("""
        SELECT v.id, a.name AS area, b.business_name, p.postcode, v.address_line, v.size
        FROM venues v
        LEFT JOIN area a ON a.id = v.area_id
        LEFT JOIN business b ON b.id = v.business_id
        LEFT JOIN postcode p ON p.id = v.postcode_id
        ORDER BY v.id LIMIT 10
    """):
        print(row)
